# Enrollment

In [ ]:
!pip install faiss-cpu

In [ ]:
WEIGHT = 'arc_r18_fp16_backbone.pth'

In [ ]:
import os
import sys
import json
import numpy as np
import faiss

# ==========================================
# 1. 경로 설정
# ==========================================
WORKING_DIR  = '/content/drive/MyDrive/Colab Notebooks/SafeAI/팀플'
if WORKING_DIR not in sys.path:
    sys.path.insert(0, WORKING_DIR)

from face_embedder import FaceEmbedder

DATASET_DIR  = '/content/drive/MyDrive/Colab Notebooks/SafeAI/팀플/데이터셋'

# 그룹별 enrollment 폴더 경로
GROUP_DIRS = {
    'korean':  os.path.join(DATASET_DIR, '한국 연예인_전처리', '한국연예인_Enrollment'),
    'western': os.path.join(DATASET_DIR, 'ms1mv3', 'MS1MV3_Enrollment'),
}

# WEIGHT_PATH        = os.path.join(WORKING_DIR, 'arc_r18_fp16_backbone.pth')
OUTPUT_INDEX_PATH  = os.path.join(WORKING_DIR, 'face_database.index')
OUTPUT_META_PATH   = os.path.join(WORKING_DIR, 'face_metadata.json')


# ==========================================
# 2. 지문 벡터 생성
# ==========================================
def make_fingerprint_vector(embeddings):
    """여러 임베딩 평균 후 L2 정규화 → 지문 벡터 1개"""
    mean_vec = np.mean(embeddings, axis=0)
    norm     = np.linalg.norm(mean_vec)
    return mean_vec / (norm + 1e-10)


# ==========================================
# 3. FAISS DB 구축
# ==========================================
def build_database(weight):
    WEIGHT_PATH = os.path.join(WORKING_DIR, weight)
    print('⏳ 모델 로딩 중...')
    embedder = FaceEmbedder(weight_path=WEIGHT_PATH)
    print('✅ 모델 로딩 완료!\n')

    fingerprints = []   # 지문 벡터 리스트
    metadata     = {}   # {faiss_id: {'name': ..., 'group': ...}}
    current_id   = 0

    # ── 그룹별 순회
    for group, enroll_root in GROUP_DIRS.items():
        if not os.path.exists(enroll_root):
            print(f'⚠️  [{group}] 폴더 없음: {enroll_root}')
            continue

        identities = sorted([
            d for d in os.listdir(enroll_root)
            if os.path.isdir(os.path.join(enroll_root, d))
        ])
        print(f'[{group}] {len(identities)}명 발견')

        for name in identities:
            person_dir  = os.path.join(enroll_root, name)
            img_paths   = embedder.get_image_paths(person_dir)

            if not img_paths:
                print(f'  ⚠️  {name}: 이미지 없음 → 스킵')
                continue

            print(f'  ⚙️  {name}: {len(img_paths)}장 처리 중...')

            # 임베딩 추출
            embeddings = embedder.extract_batch(img_paths, batch_size=64)

            if embeddings.shape[0] == 0:
                print(f'  ⚠️  {name}: 임베딩 추출 실패 → 스킵')
                continue

            # 지문 벡터 생성
            fingerprint = make_fingerprint_vector(embeddings)
            fingerprints.append(fingerprint)

            # 메타데이터 기록
            metadata[current_id] = {'name': name, 'group': group}
            current_id += 1

    if not fingerprints:
        print('❌ 등록할 지문 벡터 없음')
        return

    # ── FAISS 인덱스 구축
    db_matrix = np.vstack(fingerprints).astype('float32')
    faiss.normalize_L2(db_matrix)   # 코사인 유사도를 위한 L2 정규화

    index = faiss.IndexFlatIP(512)
    index.add(db_matrix)

    # ── 저장
    faiss.write_index(index, OUTPUT_INDEX_PATH)

    with open(OUTPUT_META_PATH, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    # ── 결과 요약
    korean_cnt  = sum(1 for v in metadata.values() if v['group'] == 'korean')
    western_cnt = sum(1 for v in metadata.values() if v['group'] == 'western')

    print('\n' + '='*50)
    print('🎉 FAISS DB 구축 완료!')
    print(f'  총 등록 인원: {index.ntotal}명')
    print(f'  korean:       {korean_cnt}명')
    print(f'  western:      {western_cnt}명')
    print(f'  인덱스 저장:  {OUTPUT_INDEX_PATH}')
    print(f'  메타데이터:   {OUTPUT_META_PATH}')
    print('='*50)


# ==========================================
# 4. 실행
# ==========================================
if __name__ == '__main__':
    # 가중치 파일 지정 가능
    build_database(weight=WEIGHT)